In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
import numpy as np
import json
import pandas as pd
import seaborn as sns
from datetime import datetime

In [3]:
# BoTorch models and transforms
from botorch.models import SingleTaskGP, ModelListGP
from botorch.models.transforms import Normalize, Standardize

from botorch.fit import fit_gpytorch_mll
fit_gpytorch_model = fit_gpytorch_mll

# GPyTorch MLL
from gpytorch.mlls import ExactMarginalLogLikelihood

# Multi-objective Monte Carlo acquisition functions
from botorch.acquisition.multi_objective import (
    qExpectedHypervolumeImprovement,
    qLogNoisyExpectedHypervolumeImprovement,
)

# BoTorch utilities
from botorch.utils.multi_objective.box_decompositions import FastNondominatedPartitioning
from botorch.optim import optimize_acqf
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

# For Parallel Executions
from joblib import Parallel, delayed
from concurrent.futures import ThreadPoolExecutor

/home/cspark/Work/simulation_codes-working/miniforge3/envs/linac-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Nicer plotting
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12,8)
%config InlineBackend.figure_format = 'retina'

In [5]:
from astra import Astra
# load astra and generator binaries
%env ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
%env GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
!echo $ASTRA_BIN
!echo $GENERATOR_BIN

env: ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
env: GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator


In [6]:
from run_astra import *
from mobo_utils import *
from utils import *
from file_io import *
from plot_utils import *

In [7]:
# Set default tensor type to double for BoTorch
torch.set_default_dtype(torch.double)

In [8]:
# Wrapper for concurrent evaluation
executor = ThreadPoolExecutor(max_workers=12)

In [9]:
# --- Checkpointing Configuration ---
CHECKPOINT_FILE = "mobo_results/mobo_checkpoint.pt"

In [10]:
# -----------------------------------------------------------
# 0. Acquisition Function Selection (Choose one here)
# -----------------------------------------------------------
# Choose your desired acquisition function by uncommenting one line:
#acquisition_mode = 'qEHVI'
acquisition_mode = 'qLogNEHVI'
# -----------------------------------------------------------

In [ ]:
# -------------------------
# 3. Initialize Training Data
# -------------------------
print("Initializing training data...")

# Attempt to load checkpoint
checkpoint = load_checkpoint()
#ratio_list = [0.10, 0.10, 0.10, 0.10, 0.10, 0.10]
ratio_list = [0.50, 0.50, 0.50, 0.50, 0.50, 0.50]

if checkpoint:
    start_iteration = checkpoint['iteration'] + 1
    train_X = checkpoint['train_X']
    train_Y = checkpoint['train_Y']
    train_feas_mask = checkpoint['train_feas_mask']
    hypervolumes = checkpoint['hypervolumes']
    train_constraints_list = checkpoint['train_constraints_list']
    # Ensure acquisition_mode matches if loaded
    if acquisition_mode != checkpoint['acquisition_mode']:
        print(f"Warning: Acquisition mode changed from {checkpoint['acquisition_mode']} to {acquisition_mode}. "
              "Ensure this is intentional when resuming.")
    
    # Reload Astra config for initial parameters if needed (assuming it's consistent)
    A = Astra('astra.in')
    A.timeout = None
    A.verbose = False
    A.run()
    init_parameters = [
        A['solenoid:maxb(1)'], A['quadrupole:q_grad(1)'], A['quadrupole:q_grad(2)'],
        A['cavity:phi(1)'], A['cavity:phi(2)'], A['cavity:phi(4)']
    ]
    
    param_bounds_list = []
    for val, ratio in zip(init_parameters, ratio_list):
        lower_val = val * (1 - ratio)
        upper_val = val * (1 + ratio)
        param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])
    bounds = torch.tensor(param_bounds_list, dtype=torch.double).T
    input_transform = Normalize(d=bounds.shape[1], bounds=bounds)

else:
    start_iteration = 0
    A = Astra('astra.in')
    A.timeout = None
    A.verbose = False
    A.run()

    # Extract initial parameters from the Astra object (6 independent parameters)
    init_parameters = [
        A['solenoid:maxb(1)'],
        A['quadrupole:q_grad(1)'],
        A['quadrupole:q_grad(2)'],
        A['cavity:phi(1)'],
        A['cavity:phi(2)'], # Use phi(2) as the initial value for common_phi_2_3
        A['cavity:phi(3)'], # Common phase for cavity 2 & 3
        A['cavity:phi(4)'],  # Use phi(4) as the initial value for common_phi_4_5
        A['cavity:phi(5)'] # Common phase for cavity 4 & 5
    ]

    # Define bounds as a ratio around initial parameters
    num_initial_samples = 16 # Changed from 10 to 15 for better initial exploration
    # Use a list of ratios for each parameter for more granular control
    param_bounds_list = []
    for val, ratio in zip(init_parameters, ratio_list):
        lower_val = val * (1 - ratio)
        upper_val = val * (1 + ratio)
        # Ensure lower bound is always <= upper bound, especially for negative initial values
        param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])

    # Convert bounds to a PyTorch tensor, transposed
    bounds = torch.tensor(param_bounds_list, dtype=torch.double).T 

    # Define input_transform using Normalize based on the bounds
    input_transform = Normalize(d=bounds.shape[1], bounds=bounds)

    # Generate initial random samples within bounds
    train_X_list = []
    for _ in range(num_initial_samples):
        sample = [np.random.uniform(low, high) for (low, high) in param_bounds_list]
        train_X_list.append(torch.tensor(sample, dtype=torch.double))
    train_X = torch.stack(train_X_list) 

    # Evaluate initial samples in parallel using ThreadPoolExecutor
    print(f"Evaluating {num_initial_samples} initial samples in parallel...")
    eval_results = list(executor.map(evaluate_objective, train_X))
    
    # Unpack results
    train_Y_list, train_feas_list, initial_constraints_list = zip(*eval_results)
    train_Y = torch.stack(train_Y_list)  # Objectives (negated)
    train_feas_mask = torch.stack(train_feas_list) # Feasibility mask
    train_constraints_list = list(initial_constraints_list)
    # train_constraints_df = pd.DataFrame(train_constraints_list) # This is for plotting, not needed for core state

print(f"Initial/Loaded training data evaluated. Feasible samples: {train_feas_mask.sum().item()} / {train_X.shape[0]}")

Initializing training data...
No checkpoint found. Starting new optimization.
Evaluating 16 initial samples in parallel...
Running simulation with parameters: [0.2370714119254546, 1.8634484103801348, -3.4112629360817026, 53.17977406787008, -37.306435128637865, -51.51331017457584]
Running simulation with parameters: [0.21135413868090158, 1.3853612992137325, -4.0160736700748085, 49.21371439697889, -32.191488517635065, -30.044278425338163]
Running simulation with parameters: [0.15563273773565287, 1.8924052064015122, -2.3484573278164285, 28.960221328414754, -44.70970413184233, -57.54013272908231]
Running simulation with parameters: [0.25700399998415996, 1.5967591295325994, -3.3784817960438405, 30.099399070589598, -38.014334990171065, -20.120512977446126]
Running simulation with parameters: [0.15767113899456042, 1.471592201233691, -3.9585678759164193, 26.890030256629064, -26.792945950738144, -27.181913619376886]
Running simulation with parameters: [0.12644345942262264, 0.9286831254110598, -

In [ ]:
# -------------------------
# BO Loop with Hypervolume Tracking
# -------------------------
hypervolumes = hypervolumes if 'hypervolumes' in locals() else [] # Initialize if not loaded
n_iterations = 300 # Total number of Bayesian Optimization iterations
q = 8             # Number of candidate points to generate in parallel per iteration

print(f"\nStarting Bayesian Optimization loop for {n_iterations} iterations (resuming from {start_iteration})...")

for iteration in range(start_iteration, n_iterations):
    print(f"\n--- Iteration {iteration + 1}/{n_iterations} ---")
    
    # Filter for feasible data to train the GP models
    feasible_X = train_X[train_feas_mask]
    feasible_Y = train_Y[train_feas_mask]

    # If no feasible points yet, use all data (or handle as appropriate for your problem)
    if feasible_X.shape[0] == 0:
        print(f"Warning: No feasible data points found yet. Using all data for GP training in iteration {iteration + 1}.")
        feasible_X = train_X
        feasible_Y = train_Y
    
    # Fit GP models for each objective on feasible data
    gp_emit_x = SingleTaskGP(feasible_X, feasible_Y[:, 0:1], input_transform=input_transform, outcome_transform=Standardize(m=1))
    gp_emit_y = SingleTaskGP(feasible_X, feasible_Y[:, 1:2], input_transform=input_transform, outcome_transform=Standardize(m=1))
    gp_energy = SingleTaskGP(feasible_X, feasible_Y[:, 2:3], input_transform=input_transform, outcome_transform=Standardize(m=1))
    
    # Combine into a ModelListGP for multi-objective optimization
    model = ModelListGP(gp_emit_x, gp_emit_y, gp_energy)

    # Fit each model in the ModelListGP
    for m in model.models:
        mll = ExactMarginalLogLikelihood(m.likelihood, m)
        fit_gpytorch_model(mll)
    print("Models fitted successfully for this iteration.")
    
    # Compute reference point for NEGATED objectives (based on feasible data)
    ref_point = compute_ref_point(feasible_Y)
    print(f"  Reference Point (Negated): {ref_point.tolist()}")

    # Partitioning for EHVI (uses feasible data)
    partitioning = FastNondominatedPartitioning(ref_point=ref_point, Y=feasible_Y)

    # Instantiate the selected acquisition function
    if acquisition_mode == 'qEHVI':
        acq_func = qExpectedHypervolumeImprovement(
            model=model,
            ref_point=ref_point.tolist(), # ref_point for acq_func is expected as list/tuple
            partitioning=partitioning,
            sampler=None, # Use default sampler
            # prune_baseline=True # Not a direct argument for qEHVI
        )
    elif acquisition_mode == 'qLogNEHVI':
        acq_func = qLogNoisyExpectedHypervolumeImprovement(
            model=model,
            ref_point=ref_point.tolist(), # ref_point for acq_func is expected as list/tuple
            X_baseline=feasible_X, # X_baseline should be feasible points for qLogNEHVI
            sampler=None, # Use default sampler
            prune_baseline=True # Recommended for performance
        )
    else:
        raise ValueError(f"Unsupported acquisition_mode: {acquisition_mode}. Choose 'qEHVI' or 'qLogNEHVI'.")

    # Optimize acquisition function to find next candidates
    print(f"Optimizing acquisition to find {q} new candidates...")
    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=bounds, # Bounds are still in the original unnormalized space
        q=q,
        num_restarts=20, # Increased for robustness
        raw_samples=128,  # Increased for robustness
        return_best_only=True
    )
    print(f"Generated candidates: {candidates.tolist()}")

    # Evaluate candidates in parallel using ThreadPoolExecutor
    print("Evaluating candidates...")
    eval_results = list(executor.map(evaluate_objective, candidates))
    
    # Unpack results
    new_Y_list, new_feas_list, new_constraints_list_tuple = zip(*eval_results) # Renamed to avoid confusion
    new_Y = torch.stack(new_Y_list)
    new_feas_mask = torch.stack(new_feas_list) # Ensure it's a tensor of booleans

    # Compute constraint violation magnitudes (for diagnostics/analysis)
    def constraint_violation(diag):
        return {
            'v_sigma_x': max(0.0, diag['sigma_x'] - 1.0e-3),
            'v_sigma_y': max(0.0, diag['sigma_y'] - 1.0e-3),
            'v_sigma_xp': max(0.0, diag['sigma_xp'] - 1.0e-3),
            'v_sigma_yp': max(0.0, diag['sigma_yp'] - 1.0e-3),
            'v_sigma_z': max(0.0, diag['sigma_z'] - 1.0e-3),
            'v_E_low': max(0.0, 195e6 - diag['mean_kinetic_energy']),
            'v_E_high': max(0.0, diag['mean_kinetic_energy'] - 205e6)
        }
    violation_dicts = [constraint_violation(d) for d in new_constraints_list_tuple] # Use the new tuple name
    df_violations = pd.DataFrame(violation_dicts) # Can be used for further analysis

    # --- Update training data ---
    train_X = torch.cat([train_X, candidates])
    train_Y = torch.cat([train_Y, new_Y])
    train_feas_mask = torch.cat([train_feas_mask, new_feas_mask]) # Concatenate boolean tensors
    # Accumulate all constraint diagnostics
    train_constraints_list.extend(list(new_constraints_list_tuple)) # Convert to list before extending
    
    print(f"Feasible in batch: {new_feas_mask.sum().item()} / {q}")
    print(f"Total points: {train_X.shape[0]}")

    # Track hypervolume
    # Calculate feasible Pareto front from all *feasible* observed points
    feasible_Y_current = train_Y[train_feas_mask]
    if feasible_Y_current.shape[0] > 0:
        pareto_mask_feasible = is_non_dominated(feasible_Y_current)
        pareto_Y_feasible = feasible_Y_current[pareto_mask_feasible]
        
        hv_calculator = Hypervolume(ref_point=ref_point)
        current_hv = hv_calculator.compute(pareto_Y_feasible)
        hypervolumes.append(current_hv)
        print(f"Current Feasible Hypervolume: {current_hv:.4f}")
    else:
        # If no feasible points, hypervolume is 0
        hypervolumes.append(0.0)
        print("No feasible points found yet, Hypervolume is 0.0.")

    # --- Save checkpoint after each iteration ---
    save_checkpoint(iteration, train_X, train_Y, train_feas_mask, hypervolumes, 
                    acquisition_mode, train_constraints_list)


print("\nBayesian Optimization loop finished.")

In [ ]:
# Save final results (negated objectives)
save_results(train_X, train_Y)

In [ ]:
# --- Plot Hypervolume vs Iteration Plot
print("Generating Hypervolume progress plot...")
plot_hypervolume(hypervolumes, n_iterations, start_iteration)

In [ ]:
# Generate 2D projection plots of the objective space
print("\nGenerating final Pareto front 2D projection plots...")
# Updated call to pass only train_Y as the function now calculates its own plotting data
plot_pareto_objective_space(train_Y)

In [ ]:
# --- Plot all constraints ---
print("Generating constraint space plots...")
# train_constraints_df is not updated in the loop, so recreate it from train_constraints_list for plotting
final_train_constraints_df = pd.DataFrame(train_constraints_list)
plot_all_constraints(final_train_constraints_df, train_feas_mask)

In [ ]:
# --- Plot objective evolution ---
print("\nGenerating objective evolution plots...")
plot_objective_evolution(train_Y, n_iterations, start_iteration, num_initial_samples, q)